# Bootstrap Quantile Regression — Versi Perbaikan

Notebook ini adalah versi terkoreksi dari skrip simulasi bootstrap untuk regresi kuantil,
beserta aplikasi backtesting Value-at-Risk pada saham Indonesia.

Setiap perbaikan diberi tanda **[FIX n]** dan dijelaskan tepat sebelum kode yang bersangkutan.

| # | Perbaikan |
|---|---|
| 1 | `calibrate_dgp`: ketidakcocokan skala kovariat yang membuat $\sigma$ meledak (penyebab estimasi berorde $10^5$–$10^6$) |
| 2 | bias / varians / RMSE dihitung dari penduga titik, bukan dari replikat bootstrap |
| 3 | varians bootstrap dilaporkan terpisah, lengkap dengan rasio terhadap varians Monte Carlo |
| 4 | wild bootstrap yang valid untuk $\tau \neq 0.5$ (Feng, He & Hu, 2011) |
| 5 | deteksi ketidakstabilan numerik yang benar-benar menyala |
| 6 | backtest VaR: rolling window, dan `adaptive` tidak lagi memakai titik tengah selang sebagai penduga |
| 7 | uji Kupiec dan Christoffersen yang benar (`chi2.sf`, $LR_{cc}=LR_{uc}+LR_{ind}$, df=2) |
| 8 | seed eksplisit sehingga hasil reproducible |
| 9 | path file dapat dikonfigurasi + fallback data sintetis |
| 10 | figur yang informatif (skala log, plot rasio varians vs coverage) |

**Cara pakai:** jalankan sel berurutan dari atas. Atur `CONFIG` di sel berikutnya.
Mulailah dengan `TEST_MODE = True` untuk memastikan semuanya jalan (sekitar 1–2 menit),
baru ubah ke `False` untuk simulasi penuh.

## 0. Setup dan konfigurasi

In [ ]:
# Jalankan sekali kalau paket belum terpasang:
# !pip install numpy pandas scipy statsmodels scikit-learn joblib matplotlib

In [ ]:
import json, os, time, warnings

import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from scipy.stats import chi2, gaussian_kde
from sklearn.linear_model import LinearRegression
from statsmodels.regression.quantile_regression import QuantReg

warnings.filterwarnings("ignore")
%matplotlib inline
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

print("numpy", np.__version__, "| pandas", pd.__version__, "| statsmodels", sm.__version__)

In [ ]:
CONFIG = {
    # True  = cek cepat (n kecil, sedikit replikasi). Pakai ini dulu.
    # False = simulasi penuh sesuai makalah (1000 replikasi, B=500).
    "TEST_MODE": True,

    # Folder berisi CSV harga. Set None untuk memakai data sintetis.
    # Contoh Windows: r"C:\Users\Dhea\Downloads"
    "DATA_DIR": None,

    # "stylized"   = DGP persis seperti tertulis di Section 2.3 makalah (disarankan)
    # "calibrated" = dikalibrasi dari data, dengan perbaikan skala [FIX 1]
    "DGP_MODE": "stylized",

    "SEED": 20260825,      # [FIX 8]

    # Di Windows/Jupyter, kalau joblib bermasalah, set ke 1.
    "N_JOBS": 4,

    "OUTDIR": "output",
}

# Ambang untuk menandai satu penduga sebagai tidak stabil secara numerik [FIX 5].
UNSTABLE_THRESHOLD = 1e8

os.makedirs(CONFIG["OUTDIR"], exist_ok=True)
METHODS = ["pairs", "residual", "wild", "wild_nofs", "wild_mammen",
           "wild_cluster", "adaptive"]
CONFIG

## 1. Pemuat data

`load_price_csv` membaca CSV harga (format Investing.com) dan menghitung log return harian.
`make_synthetic_panel` adalah fallback **[FIX 9]** supaya notebook tetap bisa dijalankan dan
diverifikasi orang lain tanpa akses ke file CSV Anda — penting untuk replikasi oleh reviewer.

In [ ]:
def load_price_csv(filepath, date_col="Date", price_col="Price"):
    """Membaca CSV harga lokal dan menghitung log return harian."""
    if not os.path.exists(filepath):
        print(f"  [!] File tidak ditemukan: {filepath}")
        return pd.DataFrame()

    df = pd.read_csv(filepath, thousands=",")
    if date_col not in df.columns or price_col not in df.columns:
        print(f"  [!] Kolom '{date_col}'/'{price_col}' tidak ada di {filepath}")
        return pd.DataFrame()

    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).sort_values(date_col).reset_index(drop=True)
    df["return"] = np.log(df[price_col] / df[price_col].shift(1))
    df = df.dropna(subset=["return"])
    return df[[date_col, "return"]].rename(columns={date_col: "Date"})


def make_synthetic_panel(n=2800, seed=0):
    """[FIX 9] Data sintetis pengganti kalau CSV tidak tersedia."""
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2014-01-01", periods=n)
    mkt = stats.t.rvs(df=5, size=n, random_state=rng) * 0.012 / np.sqrt(5 / 3)
    out = {"IHSG": pd.DataFrame({"Date": dates, "return": mkt})}
    for name, beta, idio in [("ITMG", 1.30, 0.028),
                             ("BBCA", 0.95, 0.014),
                             ("INDF", 1.05, 0.018)]:
        eps = stats.t.rvs(df=4, size=n, random_state=rng) * idio / np.sqrt(2)
        out[name] = pd.DataFrame({"Date": dates, "return": beta * mkt + eps})
    return out

print("OK")

## 2. Data Generating Process

### [FIX 1] Ketidakcocokan skala kovariat — penyebab utama angka $10^5$–$10^6$

Pada versi lama, `gamma1` diestimasi dengan meregresikan $\log|\text{residual}|$ terhadap
$|\text{return pasar}|$, variabel yang rentangnya hanya sekitar $0$ sampai $0{,}05$.
Koefisien hasilnya karena itu berorde puluhan — satuannya "per unit return".

Koefisien itu lalu dipakai pada `X = np.random.uniform(-1, 1)`, variabel yang **20–80 kali
lebih lebar skalanya**. Akibatnya

$$\sigma(X) = \exp(\gamma_0 + \gamma_1 |X|)$$

membentang belasan orde besaran di dalam **satu** sampel. Itulah sumber estimasi slope
berorde $10^5$–$10^6$ dan varians $10^{11}$ di tabel lama.

**Perbaikan:** `X` dibangkitkan dengan simpangan baku `x_scale`, yaitu simpangan baku return
pasar empiris, sehingga `gamma1` dipakai pada skala tempat ia diestimasi. Ditambah pengaman
`max_sigma_ratio` yang memangkas `gamma1` dan mencetak peringatan kalau rentang $\sigma$
melampaui batas wajar — supaya kegagalan terlihat, bukan diam-diam menjalar ke tabel hasil.

In [ ]:
def generate_skewed_t(n, nu, lam, rng):
    """Skewed-t terstandardisasi (mean 0, varians 1)."""
    e = stats.t.rvs(df=nu, size=n, random_state=rng)
    e = np.where(e < 0, e / (1 - lam), e / (1 + lam))
    e = e - np.mean(e)
    sd = np.std(e)
    return e / sd if sd > 0 else e


def generate_dgp_data(n, params, rng):
    """[FIX 1] X dibangkitkan pada skala yang sama dengan kovariat kalibrasi."""
    half_width = np.sqrt(3.0) * params["x_scale"]   # sd(Uniform(-a, a)) = a / sqrt(3)
    X = rng.uniform(-half_width, half_width, size=n)
    eps = generate_skewed_t(n, params["nu"], params["lam"], rng)
    sigma = np.exp(params["gamma0"] + params["gamma1"] * np.abs(X))
    y = params["beta0"] + params["beta1"] * X + sigma * eps
    return X, y


def stylized_dgp_params():
    """DGP terstilisasi persis seperti tertulis di Section 2.3 makalah."""
    return {"nu": 4.0, "lam": -0.4, "gamma0": 0.2, "gamma1": 0.3,
            "x_scale": 1.0 / np.sqrt(3.0), "beta0": 0.0, "beta1": 0.8}


def calibrate_dgp(stock_returns, market_returns, max_sigma_ratio=20.0):
    """[FIX 1] Kalibrasi yang konsisten skala, dengan pengaman eksplisit."""
    Xd = sm.add_constant(market_returns)
    fit = QuantReg(stock_returns, Xd).fit(q=0.5)
    residuals = stock_returns - fit.predict(Xd)

    k = stats.kurtosis(residuals, fisher=True)
    nu_est = float(max(4.1, min(30.0, 4.0 + 6.0 / k))) if k > 0 else 10.0
    lam_est = float(np.clip(stats.skew(residuals) / 1.5, -0.8, 0.8))

    abs_mkt = np.abs(market_returns)
    log_abs_resid = np.log(np.abs(residuals) + 1e-8)
    lr = LinearRegression().fit(abs_mkt.reshape(-1, 1), log_abs_resid)
    gamma0, gamma1 = float(lr.intercept_), float(lr.coef_[0])

    x_scale = float(np.std(market_returns))
    half_width = np.sqrt(3.0) * x_scale
    sigma_ratio = np.exp(abs(gamma1) * half_width)
    if sigma_ratio > max_sigma_ratio:
        gamma1_new = np.sign(gamma1) * np.log(max_sigma_ratio) / half_width
        print(f"  [!] gamma1={gamma1:.2f} menghasilkan rentang sigma {sigma_ratio:.3g}x "
              f"dalam satu sampel; dipangkas ke {gamma1_new:.2f} "
              f"(rentang {max_sigma_ratio:.0f}x).")
        gamma1 = float(gamma1_new)

    return {"nu": nu_est, "lam": lam_est, "gamma0": gamma0, "gamma1": gamma1,
            "x_scale": x_scale,
            "beta0": float(fit.params[0]), "beta1": float(fit.params[1])}

print("OK")

### Demonstrasi bug skala

Sel di bawah mereproduksi bug lama secara langsung, memakai DGP versi lama
(`X ~ Uniform(-1, 1)` dengan `gamma1` hasil kalibrasi). Jalankan untuk melihat sendiri
bagaimana estimasi slope berpindah dari sekitar $0{,}8$ ke orde $10^5$.

Sel ini murni diagnostik — boleh dilewati kalau tidak perlu.

In [ ]:
def _qr_slope_old_dgp(gamma0, gamma1, tau, n=250, reps=150, seed=1):
    """DGP versi LAMA: X ~ Uniform(-1, 1), tanpa penyesuaian skala."""
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(reps):
        X = rng.uniform(-1, 1, n)
        eps = generate_skewed_t(n, 4.2, -0.4, rng)
        y = 0.8 * X + np.exp(gamma0 + gamma1 * np.abs(X)) * eps
        try:
            out.append(QuantReg(y, sm.add_constant(X)).fit(q=tau, max_iter=1000).params[1])
        except Exception:
            pass
    return np.array(out)

print("beta1 sejati = 0.8. Estimasi seharusnya berada di sekitar nilai itu.\n")
rows = []
for label, g0, g1 in [("gamma0=0.2, gamma1=0.3  (seperti di paper)", 0.2, 0.3),
                      ("gamma0=-4.76, gamma1=12 (hasil kalibrasi)", -4.76, 12.0),
                      ("gamma0=-4.76, gamma1=20", -4.76, 20.0)]:
    for tau in [0.05, 0.5, 0.95]:
        s = _qr_slope_old_dgp(g0, g1, tau)
        rows.append({"parameter": label, "tau": tau,
                     "median |b1|": np.median(np.abs(s)), "SD": s.std()})
display(pd.DataFrame(rows).pivot(index="parameter", columns="tau",
                                 values="median |b1|").round(4))

## 3. Metode bootstrap

### [FIX 5] Deteksi ketidakstabilan numerik

Versi lama hanya menangkap *exception* dan `NaN`. `QuantReg` jarang melempar exception:
pada desain yang hampir singular ia mengembalikan angka berhingga tetapi absurd.
Akibatnya `overflow_rate` selalu bernilai `0.0` dan figur stabilitas keluar kosong.
Sekarang magnitudo koefisien ikut diperiksa.

In [ ]:
def qr_fit_safe(X, y, tau=0.5):
    """Mengembalikan (koefisien, flag_stabil)."""
    try:
        fit = QuantReg(y, sm.add_constant(X)).fit(q=tau, max_iter=2000)
        coef = np.asarray(fit.params, dtype=float)
    except Exception:
        return np.array([np.nan, np.nan]), False

    # [FIX 5] magnitudo ikut diperiksa, bukan cuma NaN/exception
    stable = bool(np.all(np.isfinite(coef)) and
                  np.max(np.abs(coef)) < UNSTABLE_THRESHOLD)
    return coef, stable


def _summarize(coefs, flags, alpha):
    """Selang persentil + proporsi replikat bootstrap yang tidak stabil."""
    unstable_rate = float(1.0 - np.mean(flags)) if len(flags) else 1.0
    valid = coefs[np.asarray(flags, dtype=bool)]
    if len(valid) < 2:
        nan2 = np.array([np.nan, np.nan])
        return coefs, nan2, nan2, 1.0
    lower = np.percentile(valid, 100 * alpha / 2, axis=0)
    upper = np.percentile(valid, 100 * (1 - alpha / 2), axis=0)
    return coefs, lower, upper, unstable_rate

print("OK")

In [ ]:
def pairs_bootstrap(X, y, coef_hat, tau, B, alpha, rng, **kw):
    n = len(X)
    coefs, flags = np.empty((B, 2)), np.empty(B, dtype=bool)
    for b in range(B):
        idx = rng.integers(0, n, size=n)
        coefs[b], flags[b] = qr_fit_safe(X[idx], y[idx], tau)
    return _summarize(coefs, flags, alpha)


def residual_bootstrap(X, y, coef_hat, tau, B, alpha, rng, **kw):
    """Catatan: skema ini MENGASUMSIKAN homoskedastisitas. Di bawah DGP
    heteroskedastik ia tidak konsisten — itu justru prediksi teoretis yang
    ingin diuji makalah ini, jadi dibiarkan apa adanya."""
    n = len(X)
    y_pred = sm.add_constant(X) @ coef_hat
    resid = y - y_pred
    coefs, flags = np.empty((B, 2)), np.empty(B, dtype=bool)
    for b in range(B):
        r_b = rng.choice(resid, size=n, replace=True)
        coefs[b], flags[b] = qr_fit_safe(X, y_pred + r_b, tau)
    return _summarize(coefs, flags, alpha)

print("OK")

### [FIX 4] Wild bootstrap Feng, He & Hu (2011) — versi yang BENAR

**Koreksi terhadap versi notebook sebelumnya.** Setelah membaca paper aslinya, implementasi
sebelumnya terbukti salah. Spesifikasi FHH yang sesungguhnya (Section 2, langkah 1–5):

- **Langkah 2:** $e_i^* = w_i |\hat e_i|$ — memakai **residual mutlak**, bukan residual bersigna.
- **Langkah 3:** $y_i^* = x_i'\hat\beta + e_i^*$.

Bobot harus memenuhi dua syarat sekaligus:

- **Kondisi 5:** $G(0) = \tau$ — kuantil ke-$\tau$ dari bobot adalah nol.
- **Kondisi 4:** $\int_0^{\infty} w^{-1}dG(w) = -\int_{-\infty}^{0} w^{-1}dG(w) = 1/2$.

Distribusi dua-titik yang disebut FHH: $w = 2(1-\tau)$ berpeluang $1-\tau$, dan $w = -2\tau$
berpeluang $\tau$.

**Di mana versi lama salah.** Sebelumnya dipakai $w=+2\tau$ berpeluang $1-\tau$ dan
$w=-2(1-\tau)$ berpeluang $\tau$ — magnitudonya tertukar. Bobot itu memenuhi Kondisi 5 tetapi
**melanggar Kondisi 4**: pada $\tau=0{,}05$ diperoleh $\int_0^\infty w^{-1}dG = 9{,}50$, bukan
$0{,}5$ — meleset 19 kali lipat. Kondisi 4 justru yang mengendalikan skala varians bootstrap,
sehingga pelanggaran itu adalah sumber over-dispersi $R=10$–$16$ yang sebelumnya keliru
dilaporkan sebagai sifat metode FHH. Pada $\tau=0{,}5$ kedua versi kebetulan bertepatan — itulah
sebabnya hasil di median dulu terlihat wajar.

**Koreksi sampel-hingga** (FHH Section 3) sebelumnya tidak diimplementasikan sama sekali.
Residual diganti dengan
$$\hat e_i + \{\hat f(0)\}^{-1} h_i \psi_\tau(\hat e_i), \qquad
h_i = x_i'\Big(\textstyle\sum_k x_k x_k'\Big)^{-1} x_i,$$
dengan $\psi_\tau(u) = \tau - I(u<0)$. Motivasinya representasi Bahadur: residual sedikit kurang
menyebar dibanding galat model. Kini aktif lewat `finite_sample_correction=True`, dan varian
`wild_nofs` disediakan untuk mengukur efek koreksi itu sendiri.

Kerapatan kontinu persamaan (3) di FHH juga diimplementasikan (`weights="continuous"`), berlaku
untuk $1/8 < \tau < 7/8$. **Bobot Mammen juga diperbaiki.** Versi sebelumnya memakai $w=\pm1$ dengan $P(w=+1)=0{,}7236$, yang punya
$E[w]=+0{,}447$ — bukan nol, dan karena itu bukan distribusi Mammen. Yang benar adalah dua-titik
$w=-(\sqrt5-1)/2$ berpeluang $(\sqrt5+1)/(2\sqrt5)$ dan $w=(\sqrt5+1)/2$ berpeluang $(\sqrt5-1)/(2\sqrt5)$,
sehingga $E[w]=0$ dan $E[w^2]=E[w^3]=1$. Skema ini tetap dipertahankan sebagai `wild_mammen`, yaitu
pembanding yang memang tidak valid untuk $\tau \neq 0{,}5$.

In [ ]:
def _fhh_weights_2pt(n, tau, rng):
    """
    [FIX 4a] Bobot dua-titik Feng, He & Hu (2011), Section 2:
    "the two-point mass distribution with probabilities 1 - tau and tau at
     w = 2(1 - tau) and -2tau, respectively."

    Memenuhi Kondisi 5 (G(0) = tau) DAN Kondisi 4
    (int_0^inf w^-1 dG = -int_-inf^0 w^-1 dG = 1/2).
    """
    return np.where(rng.random(n) < tau, -2.0 * tau, 2.0 * (1.0 - tau))


def _fhh_weights_continuous(n, tau, rng):
    """
    [FIX 4b] Kerapatan kontinu persamaan (3) di FHH, berlaku untuk 1/8 < tau < 7/8:
        g(w) = -w  pada [-2tau - 1/4, -2tau + 1/4]
        g(w) =  w  pada [2(1-tau) - 1/4, 2(1-tau) + 1/4]
    Disampel dengan inversi CDF eksak.
    """
    if not (0.125 < tau < 0.875):
        return _fhh_weights_2pt(n, tau, rng)
    u = rng.random(n)
    neg = u < tau
    k = int(neg.sum())
    a = -2.0 * tau - 0.25
    c = 2.0 * (1.0 - tau) - 0.25
    w = np.empty(n)
    w[neg] = -np.sqrt(a * a - 2.0 * tau * rng.random(k))
    w[~neg] = np.sqrt(c * c + 2.0 * (1.0 - tau) * rng.random(n - k))
    return w


def _fhh_finite_sample_correction(X, resid, tau):
    """
    [FIX 4c] Koreksi sampel-hingga FHH, Section 3. Residual e_hat_i diganti dengan

        e_hat_i + {f_hat(0)}^-1 * h_i * psi_tau(e_hat_i),

    dengan h_i = x_i' (sum_k x_k x_k')^-1 x_i dan psi_tau(u) = tau - I(u < 0).
    Motivasinya adalah representasi Bahadur: residual sedikit kurang menyebar
    dibanding galat model. FHH memakai kernel adaptif (fungsi akj di quantreg);
    di sini dipakai gaussian_kde, yang cukup memadai untuk keperluan ini.
    """
    Xd = sm.add_constant(X)
    try:
        H = Xd @ np.linalg.pinv(Xd.T @ Xd) @ Xd.T
        h = np.clip(np.diag(H), 0.0, 1.0)
        f0 = float(gaussian_kde(resid)(0.0)[0])
    except Exception:
        return resid
    if not np.isfinite(f0) or f0 < 1e-6:
        return resid
    psi = tau - (resid < 0).astype(float)
    return resid + h * psi / f0


def wild_bootstrap_qr(X, y, coef_hat, tau, B, alpha, rng,
                      weights="2pt", finite_sample_correction=True, **kw):
    """
    [FIX 4] Wild bootstrap Feng, He & Hu (2011), Section 2, langkah 1-5.

    Langkah 2: e*_i = w_i |e_hat_i|  -- residual MUTLAK, bukan residual bersigna.
    Langkah 3: y*_i = x_i' beta_hat + e*_i.

    CATATAN atas kesalahan versi sebelumnya. Sebelumnya dipakai bobot
    w = +2tau berpeluang 1-tau dan w = -2(1-tau) berpeluang tau. Bobot itu
    memenuhi Kondisi 5 (G(0) = tau) tetapi MELANGGAR Kondisi 4: untuk
    tau = 0.05 diperoleh int_0^inf w^-1 dG = 9.50, bukan 0.5 -- meleset
    19 kali lipat. Kondisi 4 justru yang mengendalikan skala varians
    bootstrap, sehingga pelanggaran itulah sumber over-dispersi (R = 10-16)
    yang sebelumnya keliru dilaporkan sebagai sifat metode FHH. Pada
    tau = 0.5 kedua versi kebetulan bertepatan, yang menjelaskan kenapa
    hasil di median dulu terlihat wajar.
    """
    n = len(X)
    y_pred = sm.add_constant(X) @ coef_hat
    resid = y - y_pred
    if finite_sample_correction:
        resid = _fhh_finite_sample_correction(X, resid, tau)
    abs_resid = np.abs(resid)
    wfun = _fhh_weights_continuous if weights == "continuous" else _fhh_weights_2pt

    coefs, flags = np.empty((B, 2)), np.empty(B, dtype=bool)
    for b in range(B):
        w = wfun(n, tau, rng)
        coefs[b], flags[b] = qr_fit_safe(X, y_pred + w * abs_resid, tau)
    return _summarize(coefs, flags, alpha)


def wild_bootstrap_qr_nofs(X, y, coef_hat, tau, B, alpha, rng, **kw):
    """Sama seperti di atas tetapi TANPA koreksi sampel-hingga, supaya efek
    koreksi itu sendiri bisa dikuantifikasi dan dilaporkan."""
    return wild_bootstrap_qr(X, y, coef_hat, tau, B, alpha, rng,
                             weights="2pt", finite_sample_correction=False)


def wild_bootstrap_mammen(X, y, coef_hat, tau, B, alpha, rng, **kw):
    """
    [FIX 11] Distribusi dua-titik Mammen (1993) yang BENAR.

    Versi sebelumnya memakai w = +/-1 dengan P(w=+1) = (sqrt(5)+1)/(2 sqrt(5)).
    Bobot itu punya E[w] = +0.447, BUKAN nol, sehingga bukan distribusi Mammen
    dan bahkan tidak memenuhi syarat paling dasar bagi bobot wild bootstrap.
    Mean tak nol menggeser seluruh respons bootstrap secara sistematis, dan
    pergeseran itu -- bukan pelanggaran kendala kuantil -- yang kemungkinan
    besar menghasilkan pola R asimetris pada hasil sebelumnya.

    Mammen (1993) yang benar:
        w = -(sqrt(5)-1)/2 = -0.6180  berpeluang (sqrt(5)+1)/(2 sqrt(5)) = 0.7236
        w = +(sqrt(5)+1)/2 = +1.6180  berpeluang (sqrt(5)-1)/(2 sqrt(5)) = 0.2764
    sehingga E[w] = 0 dan E[w^2] = E[w^3] = 1.

    Skema ini tetap dipertahankan sebagai PEMBANDING YANG TIDAK VALID untuk
    regresi kuantil: bobot Mammen dirancang untuk skor linear, dan tidak
    memenuhi Kondisi 4 maupun Kondisi 5 di Feng, He & Hu (2011) saat
    tau != 0.5.
    """
    n = len(X)
    y_pred = sm.add_constant(X) @ coef_hat
    resid = y - y_pred
    w_lo = -(np.sqrt(5.0) - 1.0) / 2.0
    w_hi = (np.sqrt(5.0) + 1.0) / 2.0
    p_lo = (np.sqrt(5.0) + 1.0) / (2.0 * np.sqrt(5.0))

    coefs, flags = np.empty((B, 2)), np.empty(B, dtype=bool)
    for b in range(B):
        w = np.where(rng.random(n) < p_lo, w_lo, w_hi)
        coefs[b], flags[b] = qr_fit_safe(X, y_pred + resid * w, tau)
    return _summarize(coefs, flags, alpha)


def wild_cluster_bootstrap(X, y, coef_hat, tau, B, alpha, rng, block_size=5,
                           finite_sample_correction=True, **kw):
    """Bobot FHH yang benar, tetapi satu bobot dibagikan untuk tiap blok berisi
    `block_size` pengamatan berurutan.

    PERHATIAN: DGP di makalah ini membangkitkan pengamatan yang saling bebas,
    jadi tidak ada dependensi lokal untuk dipertahankan dan skema ini secara
    teoretis setara dengan wild biasa. Untuk mempertahankannya sebagai metode
    tersendiri, DGP perlu diberi dependensi serial lebih dulu.
    """
    n = len(X)
    y_pred = sm.add_constant(X) @ coef_hat
    resid = y - y_pred
    if finite_sample_correction:
        resid = _fhh_finite_sample_correction(X, resid, tau)
    abs_resid = np.abs(resid)
    n_blocks = int(np.ceil(n / block_size))

    coefs, flags = np.empty((B, 2)), np.empty(B, dtype=bool)
    for b in range(B):
        bw = _fhh_weights_2pt(n_blocks, tau, rng)
        w = np.repeat(bw, block_size)[:n]
        coefs[b], flags[b] = qr_fit_safe(X, y_pred + w * abs_resid, tau)
    return _summarize(coefs, flags, alpha)

print("OK")

## 4. Aturan adaptif (Breusch–Pagan + Hill)

In [ ]:
def breusch_pagan_pvalue(X, residuals):
    model = sm.OLS(residuals ** 2, sm.add_constant(X)).fit()
    return float(chi2.sf(len(X) * model.rsquared, df=1))


def hill_estimator(x, k=None):
    """Indeks ekor Hill, dengan penjagaan k yang eksplisit."""
    xs = np.sort(np.abs(x))[::-1]
    xs = xs[xs > 0]
    if k is None:
        k = max(2, int(len(xs) * 0.1))
    k = min(k, len(xs) - 1)
    if k < 2:
        return np.inf
    return float(1.0 / (np.mean(np.log(xs[:k] / xs[k])) + 1e-12))


def adaptive_bootstrap(X, y, coef_hat, tau, B, alpha, rng, **kw):
    resid = y - sm.add_constant(X) @ coef_hat
    is_heavy = hill_estimator(resid) < 3.0
    is_hetero = breusch_pagan_pvalue(X, resid) < 0.05
    if is_heavy:
        return residual_bootstrap(X, y, coef_hat, tau, B, alpha, rng)
    if is_hetero:
        return wild_bootstrap_qr(X, y, coef_hat, tau, B, alpha, rng)
    return pairs_bootstrap(X, y, coef_hat, tau, B, alpha, rng)


BOOTSTRAP_METHODS = {
    "pairs": pairs_bootstrap,
    "residual": residual_bootstrap,
    "wild": wild_bootstrap_qr,
    "wild_nofs": wild_bootstrap_qr_nofs,
    "wild_mammen": wild_bootstrap_mammen,
    "wild_cluster": wild_cluster_bootstrap,
    "adaptive": adaptive_bootstrap,
}
print("Metode terdaftar:", list(BOOTSTRAP_METHODS))

## 5. Simulasi Monte Carlo

### [FIX 2] Perbedaan paling penting terhadap versi lama

Versi lama menghitung, **di dalam** tiap replikasi:

```python
bias_slope = np.mean(coefs[:, 1]) - true_beta1   # coefs = replikat bootstrap
var_slope  = np.var(coefs[:, 1])                 # varians bootstrap
rmse_slope = np.sqrt(bias_slope**2 + var_slope)
```

lalu merata-ratakan ketiganya lintas replikasi Monte Carlo. Ada dua akibat fatal.

**Pertama,** bias dan RMSE jadi bergantung pada metode bootstrap — padahal keduanya adalah
sifat penduga titik $\hat\beta_1$, yang **identik** untuk kelima metode. Bootstrap hanya
mempengaruhi selang, bukan titiknya. Itu sebabnya tabel lama menampilkan RMSE $91{,}17$
(pairs) vs $9{,}85$ (residual) untuk penduga yang sama persis.

**Kedua,** identitas $\text{RMSE}^2 = \text{bias}^2 + \text{varians}$ jadi tidak berlaku
setelah perata-rataan, karena rata-rata dari akar bukan akar dari rata-rata.

**Perbaikan:** penduga titik dari fit asli disimpan per replikasi, lalu bias/varians/RMSE
dihitung dari sebarannya lintas replikasi. Varians bootstrap disimpan terpisah **[FIX 3]**,
bersama `var_ratio` — rasio yang menjelaskan coverage.

In [ ]:
def run_single_replication(seed, n, tau, dgp_params, methods, B):
    rng = np.random.default_rng(seed)
    X, y = generate_dgp_data(n, dgp_params, rng)
    true_beta1 = dgp_params["beta1"]

    coef_hat, ok = qr_fit_safe(X, y, tau)
    if not ok:
        return None

    # [FIX 2] penduga titik disimpan sekali, di luar loop metode
    out = {"beta1_hat": float(coef_hat[1])}
    for name in methods:
        fn = BOOTSTRAP_METHODS[name]
        try:
            coefs, lo, hi, unstable = fn(X, y, coef_hat, tau, B, 0.05, rng)
            if not (np.isfinite(lo[1]) and np.isfinite(hi[1])):
                out[name] = None
                continue
            valid = coefs[np.isfinite(coefs).all(axis=1)][:, 1]
            out[name] = {
                "covered": bool(lo[1] <= true_beta1 <= hi[1]),
                "width": float(hi[1] - lo[1]),
                "var_boot": float(np.var(valid, ddof=1)) if len(valid) > 1 else np.nan,
                "unstable_rate": float(unstable),
            }
        except Exception:
            out[name] = None
    return out

In [ ]:
def run_simulation(n, tau, dgp_params, methods, n_mc, B, base_seed, n_jobs):
    print(f"  n={n}, tau={tau}: {n_mc} replikasi x {B} bootstrap ...", flush=True)
    t0 = time.time()
    # [FIX 8] seed unik per replikasi -> reproducible walau paralel
    seeds = np.random.SeedSequence(base_seed).spawn(n_mc)
    reps = Parallel(n_jobs=n_jobs)(
        delayed(run_single_replication)(s, n, tau, dgp_params, methods, B)
        for s in seeds
    )
    reps = [r for r in reps if r is not None]
    if not reps:
        return {}

    true_beta1 = dgp_params["beta1"]
    beta1_hats = np.array([r["beta1_hat"] for r in reps])

    # Sifat penduga titik — SATU set angka, sama untuk semua metode.
    bias = float(np.mean(beta1_hats) - true_beta1)
    var_mc = float(np.var(beta1_hats, ddof=1))
    rmse = float(np.sqrt(np.mean((beta1_hats - true_beta1) ** 2)))

    summary = {"_point_estimator": {"bias": bias, "var_mc": var_mc, "rmse": rmse,
                                    "n_valid_reps": len(reps)}}
    for m in methods:
        vals = [r[m] for r in reps if r.get(m) is not None]
        if not vals:
            continue
        var_boot = float(np.nanmean([v["var_boot"] for v in vals]))
        summary[m] = {
            "coverage": float(np.mean([v["covered"] for v in vals])),
            "width": float(np.mean([v["width"] for v in vals])),
            "var_boot": var_boot,
            # [FIX 3] Rasio inilah yang menjelaskan coverage. Nilai jauh di bawah 1
            # berarti bootstrap meremehkan variabilitas sampling sesungguhnya
            # -> selang terlalu sempit -> undercoverage.
            "var_ratio": float(var_boot / var_mc) if var_mc > 0 else np.nan,
            "unstable_rate": float(np.mean([v["unstable_rate"] for v in vals])),
            "n_used": len(vals),
        }
    print(f"    selesai dalam {time.time()-t0:.1f}s "
          f"({len(reps)}/{len(seeds)} replikasi valid)", flush=True)
    return summary

## 6. Backtesting Value-at-Risk

### [FIX 6] Dua koreksi

**(a) Backtest tidak rolling.** Versi lama meng-fit **sekali** pada 500 observasi pertama,
lalu memakai koefisien itu untuk seluruh sisa sampel (~2293 hari, hampir 9 tahun).
Itu bukan backtest out-of-sample yang lazim, dan menjelaskan kenapa BBCA melanggar di 10%
pada **kedua** ekor. Sekarang model di-refit secara rolling.

**(b) `coef = (lower + upper) / 2`.** Versi lama memakai titik tengah selang persentil sebagai
penduga untuk `adaptive_smart`. Titik tengah selang bukan penduga; itulah satu-satunya alasan
`adaptive` berbeda dari metode lain di tabel backtesting.

Konsekuensi jujur dari (b): ramalan VaR titik **identik** untuk semua metode bootstrap, karena
semuanya memakai fit QR yang sama. Karena itu backtest pelanggaran tidak bisa — dan tidak
seharusnya — membedakan metode bootstrap. Fungsi di bawah melaporkan backtest titik satu kali
saja (netral metode), lalu membandingkan lebar selang bootstrap secara terpisah.

### [FIX 7] Uji Kupiec dan Christoffersen

Versi lama memakai `1 - chi2.cdf(...)`, yang *underflow* ke `0.0` persis untuk statistik besar —
itulah asal deretan `p = 0.0000` di tabel lama. Diganti `chi2.sf`.

Versi lama juga menghitung statistik **independensi** (df = 1) lalu melabelinya "CC". Cakupan
bersyarat yang benar adalah $LR_{cc} = LR_{uc} + LR_{ind}$ dengan df = 2. Ketiganya kini
dilaporkan terpisah: `p_uc`, `p_ind`, `p_cc`.

In [ ]:
def kupiec_lr(n_viol, n_obs, tau):
    """[FIX 7] LR uji cakupan tak bersyarat, dengan konvensi 0*log(0)=0."""
    if n_obs == 0:
        return np.nan
    p = n_viol / n_obs
    ll_null = n_viol * np.log(tau) + (n_obs - n_viol) * np.log(1 - tau)
    ll_alt = 0.0
    if n_viol > 0:
        ll_alt += n_viol * np.log(p)
    if n_viol < n_obs:
        ll_alt += (n_obs - n_viol) * np.log(1 - p)
    return float(-2.0 * (ll_null - ll_alt))


def christoffersen_lr(violations):
    """[FIX 7] LR uji INDEPENDENSI (df=1), dilaporkan terpisah dari CC."""
    v = np.asarray(violations, dtype=int)
    n00 = int(np.sum((v[:-1] == 0) & (v[1:] == 0)))
    n01 = int(np.sum((v[:-1] == 0) & (v[1:] == 1)))
    n10 = int(np.sum((v[:-1] == 1) & (v[1:] == 0)))
    n11 = int(np.sum((v[:-1] == 1) & (v[1:] == 1)))
    total = n00 + n01 + n10 + n11
    if total == 0 or (n01 + n11) == 0 or (n00 + n10) == 0:
        return np.nan
    pi = (n01 + n11) / total
    pi0 = n01 / (n00 + n01) if (n00 + n01) > 0 else 0.0
    pi1 = n11 / (n10 + n11) if (n10 + n11) > 0 else 0.0

    def _ll(a, b, p):
        out = 0.0
        if a > 0 and p < 1:
            out += a * np.log(1 - p)
        if b > 0 and p > 0:
            out += b * np.log(p)
        return out

    ll_null = _ll(n00 + n10, n01 + n11, pi)
    ll_alt = _ll(n00, n01, pi0) + _ll(n10, n11, pi1)
    return float(-2.0 * (ll_null - ll_alt))

In [ ]:
def var_backtest(stock_returns, market_returns, methods, taus=(0.05, 0.95),
                 window=500, refit_every=20, B=200, seed=0,
                 ci_every=10, max_test_points=None):
    """[FIX 6] Rolling refit; semua metode memakai fit QR yang sama."""
    rng = np.random.default_rng(seed)
    n_total = len(stock_returns)
    if n_total < window + 100:
        return None

    test_idx = np.arange(window, n_total)
    if max_test_points is not None:
        test_idx = test_idx[:max_test_points]
    var_fc = {t: np.full(len(test_idx), np.nan) for t in taus}
    boot_width = {t: {m: [] for m in methods} for t in taus}
    fits = {t: None for t in taus}

    for j, i in enumerate(test_idx):
        if j % refit_every == 0:
            Xtr = market_returns[i - window:i]
            ytr = stock_returns[i - window:i]
            for t in taus:
                coef, ok = qr_fit_safe(Xtr, ytr, t)
                fits[t] = coef if ok else None
                if ok and j % (refit_every * ci_every) == 0:
                    for m in methods:
                        try:
                            _, lo, hi, _ = BOOTSTRAP_METHODS[m](
                                Xtr, ytr, coef, t, B, 0.05, rng)
                            if np.isfinite(lo[1]) and np.isfinite(hi[1]):
                                boot_width[t][m].append(float(hi[1] - lo[1]))
                        except Exception:
                            pass
        for t in taus:
            if fits[t] is not None:
                var_fc[t][j] = fits[t][0] + fits[t][1] * market_returns[i]

    results = {}
    y_test = stock_returns[test_idx]
    for t in taus:
        ok = np.isfinite(var_fc[t])
        yt, vt = y_test[ok], var_fc[t][ok]
        viol = (yt < vt).astype(int) if t < 0.5 else (yt > vt).astype(int)
        n_obs, n_viol = len(viol), int(viol.sum())
        exp_tau = t if t < 0.5 else 1 - t

        lr_uc = kupiec_lr(n_viol, n_obs, exp_tau)
        lr_ind = christoffersen_lr(viol)
        lr_cc = lr_uc + lr_ind if np.isfinite(lr_ind) else np.nan

        results[f"tau{t}"] = {
            "n_obs": n_obs,
            "n_violations": n_viol,
            "rate": float(n_viol / n_obs),
            "expected_rate": float(exp_tau),
            # [FIX 7] chi2.sf, bukan 1 - chi2.cdf
            "p_uc": float(chi2.sf(lr_uc, df=1)) if np.isfinite(lr_uc) else np.nan,
            "p_ind": float(chi2.sf(lr_ind, df=1)) if np.isfinite(lr_ind) else np.nan,
            "p_cc": float(chi2.sf(lr_cc, df=2)) if np.isfinite(lr_cc) else np.nan,
            "mean_boot_ci_width": {m: (float(np.mean(w)) if w else np.nan)
                                   for m, w in boot_width[t].items()},
        }
    return results

## 7. Memuat data dan menetapkan DGP

In [ ]:
files = {"ITMG": "Indo Tambangraya Stock Price History.csv",
         "BBCA": "Bank Central Asia Stock Price History.csv",
         "INDF": "Indofood Stock Price History.csv",
         "IHSG": "Jakarta Stock Exchange Composite Index Historical Data.csv"}

dfs = {}
if CONFIG["DATA_DIR"]:
    print(f"Membaca CSV dari {CONFIG['DATA_DIR']} ...")
    for name, fn in files.items():
        d = load_price_csv(os.path.join(CONFIG["DATA_DIR"], fn))
        if not d.empty:
            dfs[name] = d
            print(f"  {name}: {len(d)} observasi")

if "IHSG" not in dfs:
    print("[!] Data pasar tidak tersedia -> memakai data sintetis.")
    dfs = make_synthetic_panel(seed=CONFIG["SEED"])

market_df = dfs.pop("IHSG")
common = set(market_df["Date"])
for d in dfs.values():
    common &= set(d["Date"])
common = sorted(common)
print(f"\nTanggal beririsan: {len(common)}")

mkt = market_df[market_df["Date"].isin(common)].sort_values("Date")["return"].values
stocks = {k: v[v["Date"].isin(common)].sort_values("Date")["return"].values
          for k, v in dfs.items()}
print("Saham:", list(stocks))

In [ ]:
if CONFIG["DGP_MODE"] == "calibrated" and "ITMG" in stocks:
    print("Mengkalibrasi DGP dari ITMG vs pasar ...")
    dgp = calibrate_dgp(stocks["ITMG"], mkt)
else:
    dgp = stylized_dgp_params()

print("\nParameter DGP:")
for k, v in dgp.items():
    print(f"  {k:9s} = {v: .4f}")
hw = np.sqrt(3) * dgp["x_scale"]
print(f"  -> sigma membentang {np.exp(abs(dgp['gamma1'])*hw):.2f}x dalam satu sampel")

## 8. Menjalankan simulasi

Dengan `TEST_MODE = True` ini memakan sekitar satu menit. Dengan `TEST_MODE = False`
(1000 replikasi, $B=500$, enam skenario) perkirakan beberapa jam tergantung jumlah core.

In [ ]:
if CONFIG["TEST_MODE"]:
    scenarios = [{"n": 100, "tau": t, "n_mc": 20, "B": 30} for t in (0.05, 0.5, 0.95)]
else:
    scenarios = [{"n": n, "tau": t, "n_mc": 1000, "B": 500}
                 for n in (250, 500) for t in (0.05, 0.50, 0.95)]

print("=" * 70)
print("MONTE CARLO SIMULATION")
print("=" * 70)
sim_results = {}
for i, s in enumerate(scenarios):
    key = f"n{s['n']}_tau{s['tau']:.2f}"
    sim_results[key] = run_simulation(s["n"], s["tau"], dgp, METHODS,
                                      s["n_mc"], s["B"],
                                      CONFIG["SEED"] + 1000 * i, CONFIG["N_JOBS"])

with open(os.path.join(CONFIG["OUTDIR"], "simulation_results.json"), "w") as f:
    json.dump(sim_results, f, indent=2)
print("\nDisimpan: simulation_results.json")

## 9. Tabel hasil

Dua tabel terpisah, dan pemisahan ini adalah inti **[FIX 2]**:

- **Tabel 1** — sifat penduga titik. Satu baris per skenario, karena bias/varians/RMSE tidak
  bergantung pada metode bootstrap.
- **Tabel 2** — kinerja selang. Satu baris per metode, berisi coverage, lebar, varians
  bootstrap, dan `var_ratio`.

In [ ]:
pt_rows = [{"n": int(k.split("_")[0][1:]), "tau": float(k.split("tau")[1]),
            **v["_point_estimator"]}
           for k, v in sim_results.items() if "_point_estimator" in v]
df_point = pd.DataFrame(pt_rows)
df_point.to_csv(os.path.join(CONFIG["OUTDIR"], "point_estimator_summary.csv"), index=False)

print("TABEL 1: sifat penduga titik (sama untuk semua metode bootstrap)")
display(df_point.round(6))

In [ ]:
ci_rows = []
for k, v in sim_results.items():
    n, tau = int(k.split("_")[0][1:]), float(k.split("tau")[1])
    for m in METHODS:
        if m in v:
            ci_rows.append({"n": n, "tau": tau, "method": m, **v[m]})
df_ci = pd.DataFrame(ci_rows)
df_ci.to_csv(os.path.join(CONFIG["OUTDIR"], "interval_summary.csv"), index=False)

print("TABEL 2: kinerja selang bootstrap")
display(df_ci.round(4))

## 10. Verifikasi

Sel ini memeriksa bahwa perbaikan benar-benar berlaku. Jalankan sebelum memakai hasilnya
di makalah — kalau ada yang `GAGAL`, jangan lanjut.

In [ ]:
print("VERIFIKASI 1 - identitas RMSE^2 = bias^2 + varians (dengan koreksi ddof):")
all_ok = True
for _, r in df_point.iterrows():
    m = r.n_valid_reps
    lhs = r.rmse ** 2
    rhs = r.bias ** 2 + r.var_mc * (m - 1) / m
    ok = abs(lhs - rhs) < 1e-8
    all_ok &= ok
    print(f"  n={r.n:.0f} tau={r.tau}: {lhs:.8f} vs {rhs:.8f}  {'OK' if ok else 'GAGAL'}")

print("\nVERIFIKASI 2 - bias/RMSE hanya SATU set per skenario:")
print(f"  baris tabel penduga titik = {len(df_point)} (harusnya = jumlah skenario "
      f"= {len(scenarios)})")
print(f"  kolom bias/rmse di tabel selang: "
      f"{[c for c in df_ci.columns if c in ('bias','rmse')] or 'tidak ada -> OK'}")

print("\nVERIFIKASI 3 - tidak ada magnitudo absurd:")
print(f"  |bias| maks = {df_point.bias.abs().max():.4f}")
print(f"  var_mc maks = {df_point.var_mc.max():.4f}")
print(f"  RMSE maks   = {df_point.rmse.max():.4f}   (beta1 sejati = {dgp['beta1']})")

print("\nVERIFIKASI 4 - kontras wild Mammen vs wild yang sesuai tau:")
sub = df_ci[(df_ci.tau != 0.5) & (df_ci.method.isin(["wild", "wild_mammen"]))]
display(sub[["n", "tau", "method", "var_ratio", "coverage"]].round(4))

## 11. Backtesting VaR

In [ ]:
print("=" * 70)
print("VaR BACKTESTING")
print("=" * 70)
bt = {}
for name, ret in stocks.items():
    print(f"\n-- {name} --")
    r = var_backtest(ret, mkt, METHODS,
                     window=250 if CONFIG["TEST_MODE"] else 500,
                     refit_every=20,
                     B=25 if CONFIG["TEST_MODE"] else 200,
                     seed=CONFIG["SEED"],
                     ci_every=10,
                     max_test_points=200 if CONFIG["TEST_MODE"] else None)
    if r is None:
        print("  data tidak cukup")
        continue
    bt[name] = r
    for k, v in r.items():
        print(f"  {k}: n_obs={v['n_obs']}, violations={v['n_violations']}, "
              f"rate={v['rate']:.4f} (harapan {v['expected_rate']:.2f}), "
              f"p_uc={v['p_uc']:.3g}, p_ind={v['p_ind']:.3g}, p_cc={v['p_cc']:.3g}")

with open(os.path.join(CONFIG["OUTDIR"], "backtest_results.json"), "w") as f:
    json.dump(bt, f, indent=2)

bt_rows = [{"Stock": s, "tau": k.replace("tau", ""), "n_obs": v["n_obs"],
            "Violation_Rate": v["rate"], "Expected_Rate": v["expected_rate"],
            "p_UC": v["p_uc"], "p_IND": v["p_ind"], "p_CC": v["p_cc"]}
           for s, res in bt.items() for k, v in res.items()]
df_bt = pd.DataFrame(bt_rows)
df_bt.to_csv(os.path.join(CONFIG["OUTDIR"], "backtest_summary.csv"), index=False)
display(df_bt.round(4))

## 12. Figur

**[FIX 10]** Figur `bias_variance_decomposition.png` yang lama praktis tak terbaca: batang
varians berorde $10^{11}$ menenggelamkan bias dan RMSE sampai tak terlihat, dan figur
`overflow_rates.png` keluar kosong total karena `overflow_rate` selalu nol.

Penggantinya adalah plot rasio varians terhadap coverage. Ini figur diagnostik utama:
coverage sepenuhnya dijelaskan oleh seberapa baik varians bootstrap melacak variabilitas
sampling sesungguhnya. Titik di kiri garis vertikal berarti bootstrap meremehkan variabilitas.

In [ ]:
# Nama tampilan HARUS sama persis dengan kolom "Scheme" di Tabel 2 naskah.
LABELS = {
    "pairs":        "Pairs",
    "residual":     "Residual",
    "wild":         "Wild (FHH)",
    "wild_nofs":    "Wild (FHH, no correction)",
    "wild_mammen":  "Wild (Mammen)",
    "wild_cluster": "Wild cluster",
    "adaptive":     "Adaptive",
}
MARKERS = {"pairs": "o", "residual": "s", "wild": "^", "wild_nofs": "<",
           "wild_mammen": "v", "wild_cluster": "D", "adaptive": "P"}
ORDER = ["pairs", "residual", "wild", "wild_nofs", "wild_mammen",
         "wild_cluster", "adaptive"]


def scenario_label(key):
    """'n250_tau0.05' -> label dua baris, tanpa rotasi."""
    n, tau = key.split("_tau")
    return f"$n = {n[1:]}$\n$\\tau = {float(tau):g}$"


present = [m for m in ORDER if any(m in v for v in sim_results.values())]

# --- Figur 1: rasio varians vs coverage ---
fig, ax = plt.subplots(figsize=(7.5, 5.2))
for m in present:
    xs = [v[m]["var_ratio"] for v in sim_results.values() if m in v]
    ys = [v[m]["coverage"] for v in sim_results.values() if m in v]
    ax.scatter(xs, ys, label=LABELS[m], marker=MARKERS[m],
               s=70, alpha=0.85, edgecolors="white", linewidths=0.6)
ax.axhline(0.95, ls="--", c="grey", lw=1)
ax.axvline(1.0, ls="--", c="grey", lw=1)
ax.set_xscale("log")
ax.set_xlabel("bootstrap variance / Monte Carlo variance (log scale)")
ax.set_ylabel("empirical coverage")
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.3, ls="--")
# Legenda di LUAR area plot supaya tidak menutupi titik mana pun.
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16),
          ncol=3, fontsize=9, frameon=False)
fig.savefig(os.path.join(CONFIG["OUTDIR"], "var_ratio_vs_coverage.png"),
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# --- Figur 2: coverage per sel desain ---
keys = list(sim_results.keys())
fig, ax = plt.subplots(figsize=(10, 5.2))
x = np.arange(len(keys))
w = 0.8 / max(len(present), 1)
for i, m in enumerate(present):
    ax.bar(x + i * w, [sim_results[k].get(m, {}).get("coverage", np.nan) for k in keys],
           w, label=LABELS[m])
ax.axhline(0.95, ls="--", c="black", lw=1.2)
ax.set_xticks(x + 0.4 - w / 2)
ax.set_xticklabels([scenario_label(k) for k in keys])
ax.set_ylabel("empirical coverage")
ax.set_ylim(0, 1.02)
ax.grid(axis="y", alpha=0.3, ls="--")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.10),
          ncol=4, fontsize=9, frameon=False)
fig.savefig(os.path.join(CONFIG["OUTDIR"], "coverage_by_scenario.png"),
            dpi=300, bbox_inches="tight")
plt.show()

## Catatan penutup

**Yang perlu Anda putuskan.** `wild_cluster` memblok residual untuk "mempertahankan dependensi
lokal", tapi DGP di makalah ini membangkitkan pengamatan yang saling bebas — tidak ada yang
bisa dipertahankan. Pilihannya: tambahkan dependensi serial ke DGP (misalnya error AR(1))
sehingga blok punya makna, atau keluarkan wild cluster dari makalah.

**Untuk makalah.** Klaim lama "wild bootstrap fails in tails" perlu diganti dengan klaim yang
lebih tajam dan lebih benar: bobot yang tidak disesuaikan dengan $\tau$ yang gagal, dan
koreksi Feng–He–Hu memperbaikinya. Bandingkan baris `wild` dan `wild_mammen` di Tabel 2.

**Referensi untuk [FIX 4]:** Feng, X., He, X. and Hu, J. (2011). Wild bootstrap for quantile
regression. *Biometrika*, 98(4), 995–999.